# EDACaP Subseasonal-to-Seasonal Climate Forecast Module V2

This notebook is prepared for EDACaP Seasonal Climate forecast based of IRI PyCPT Version 2 subseasonal-to-seasonal climate forecasting workflow.This notebook uses PyCPT v2 utilities to

1. download data from the IRI Data Library (through the CPT-DL python library) 
2. Run bias-correction using the IRI Climate Predictability Tool (through its companion python library, CPT-CORE) 
3. Plot skills scores and spatial loadings
4. Produce a multi-model ensemble forecast by taking the simple average of the bias-corrected members
5. Plots skill scores, deterministic forecasts, probabilistic forecasts, and exceedance probabilities for this NextGen MME forecast. 


#### Imports - This cell imports PyCPTv2 libraries 

In [9]:
import pycpt
import packaging
min_version = '2.5.0'
assert packaging.version.parse(pycpt.__version__) >= packaging.version.parse(min_version), f'This notebook requires version {min_version} or higher of the pycpt library, but you have version {pycpt.__version__}. Please close the notebook, update your environment, and load the notebook again. See https://iri-pycpt.github.io/installation/'

import cptdl as dl 
from cptextras import get_colors_bars
import datetime as dt
import numpy as np
from pathlib import Path
from pycpt import subseasonal
import xarray as xr
import sys


ModuleNotFoundError: No module named 'cptextras'

#### Define case directory
The directory where inputs, outputs, and figures generated by this notebook will be stored.

In [ ]:
# """ """ print("PARAMETERS--------------")
ini_mon = sys.argv[1]
print(ini_mon)
MOS = sys.argv[2]#'CCA' # must be one of 'CCA', 'PCR'
print(MOS)
# your selections for your predictors and predictands.
predictor_names = sys.argv[3].split(",") #[ "SEAS51.PRCP","GCFS2p1.PRCP","METEOFRANCE8.PRCP","SPEAR.PRCP", "CCSM4.PRCP"]
print(predictor_names)
predictand_name = sys.argv[4] #'UCSB.PRCP'
print(predictand_name)
predictor_extent = sys.argv[5].split(",")
predictor_extent = {
        'east': int(predictor_extent[0]),
        'west': int(predictor_extent[1]), 
        'north': int(predictor_extent[2]),
        'south': int(predictor_extent[3]),
    }
print(predictor_extent)
predictand_extent = sys.argv[6].split(",")
predictand_extent = {
        'east': int(predictand_extent[0]),
        'west': int(predictand_extent[1]), 
        'north': int(predictand_extent[2]),
        'south': int(predictand_extent[3]),
    }
print(predictand_extent)
tailoring = sys.argv[7]
tailoring = None if tailoring == 'None' else tailoring
print(tailoring)
cca_modes = sys.argv[8].split(",")
cca_modes = (int(cca_modes[0]), int(cca_modes[1]))
print(cca_modes)
x_eof_modes = sys.argv[9].split(",")
x_eof_modes = (int(x_eof_modes[0]), int(x_eof_modes[1]))
print(x_eof_modes)
y_eof_modes = sys.argv[10].split(",")
y_eof_modes = (int(y_eof_modes[0]), int(y_eof_modes[1]))
print(y_eof_modes)
scree = bool(sys.argv[11])
print(type(scree))
#sys.exit()


### Generate weekly leads for a given month based on the start date

In [1]:
import calendar
def generate_leads(start_date):
    leads = []

    # Adjust the start date based on the day of the week
    if start_date.weekday() in [5, 6]:  # Saturday or Sunday
        start_date += dt.timedelta(days=(7 - start_date.weekday()))  # Move to next Monday
    elif start_date.weekday() == 4:  # Friday
        start_date += dt.timedelta(days=3)  # Move to next Monday
    elif start_date.weekday() == 2:  # Wednesday
        start_date += dt.timedelta(days=1)  # Move to next Thursday
    elif start_date.weekday() == 1:  # Tuesday
        start_date += dt.timedelta(days=2)  # Move to next Thursday


    current_date = start_date
    week_number = 1
    last_day_of_month = calendar.monthrange(current_date.year, current_date.month)[1]

    while week_number <= 4 and current_date.month == start_date.month:
        if week_number == 1 and current_date.weekday() == 3:  # Thursday
            # Week 1 spans from Thursday to the next Sunday (10 days)
            sunday = current_date + dt.timedelta(days=10 - current_date.weekday())
            # Ensure we don't go beyond the month's end
            if sunday.month != current_date.month:
                sunday = dt.datetime(current_date.year, current_date.month, last_day_of_month)
            leads.append(('Week 1', current_date.day, sunday.day))
            week_number += 1
            current_date = sunday + dt.timedelta(days=1)
        elif current_date.weekday() == 0:  # Monday
            monday = current_date
            if week_number == 4 or (current_date + dt.timedelta(days=6)).month != current_date.month:  # Last week of the month
                sunday = dt.datetime(monday.year, monday.month, last_day_of_month)
            else:
                sunday = current_date + dt.timedelta(days=6)
            leads.append(('Week {}'.format(week_number), monday.day, sunday.day))
            week_number += 1
            current_date = sunday + dt.timedelta(days=1)
        else:
            current_date += dt.timedelta(days=1)

    return leads

In [2]:
fcst_year = dt.date.today().year
print(fcst_year)
fmonth = dt.date.today().month
print(fmonth)
fcst_date = dt.datetime(fcst_year, fmonth, 1)
fcst_mon = fcst_date.strftime("%b")  
training_seas = (dt.datetime(fcst_year,fcst_date.month-1,1).strftime("%b") + "-" + dt.datetime(fcst_year,fcst_date.month+1,1).strftime("%b"))
leads_data = generate_leads(fcst_date)

case_dir = Path.home() / "EDACaP_S2S" / str(fcst_date.year) / fcst_mon

AttributeError: 'method_descriptor' object has no attribute 'today'

#### Parameters - This cell defines the parameters of your CPT analysis

In [6]:
MOS = 'CCA'  # must be one of 'CCA', 'PCR', or "None"
predictor_names = ["GEFSv12.PRCP"]
# predictor_names = ["ECMWF.PRCP"]
predictand_name = 'CHIRPS.PRCP'
local_predictand_file = None


download_args = {
    # 'fdate':
    #   The initialization date of the model forecasts / hindcasts.
    #   This field is defined by a python datetime.datetime object,
    #   for example: dt.datetime(2022, 5, 1) # YYYY, MM, DD as integers
    #   The year field is only used for forecasts, otherwise ignored.
    #   The day field is only used in subseasonal forecasts, otherwise ignored.
    #   The month field is an integer representing a month - ie, May=5.
    'fdate':  fcst_date,  # dt.datetime(2024, 2, 1),

    # 'leads':
    #   A list of target periods, each of which is represented as (name, start, end), where
    #   start and end are numbers of days after the forecast date. For example, if
    #   fdate is 15 Jun then ('Week1', 1, 7) represents a target period of 16-22 Jun.
    'leads': leads_data,

    # 'training_season':
    #   The training set will comprise hindcasts issued within this range of months.
    #   E.g. 'May-Jul'
    'training_season': training_seas,  # 'Jan-Mar',

    # 'predictor_extent':
    #   The geographic bounding box of the climate model data you want to download.
    #   This field is defined by a python dictionary with the keys "north", "south",
    #   "east", and "west", each of which maps to a python integer representing the
    #   edge of a bounding box. i.e., "north" will be the northernmost boundary,
    #   "south" the southernmost boundary.
    #   Example: {"north": 90, "south": -90, "east": 0, "west": 180}
     'predictor_extent': predictor_extent,

    # 'predictand_extent':
    #   The geographic bounding box of the observation data you want to download.
    #   This field is defined by a python dictionary with the keys "north", "south",
    #   "east", and "west", each of which maps to a python integer representing the
    #   edge of a bounding box. i.e., "north" will be the northernmost boundary,
    #   "south" the southernmost boundary.
    #   Example: {"north": 90, "south": -90, "east": 0, "west": 180}
     'predictand_extent': predictand_extent,
    # 'filetype':
    #   The filetype to be downloaded. for now, it saves a lot of headache just to set this equal
    #   to 'cptv10.tsv' which is a boutique plain-text CPT filetype based on .tsv + metadata.
    'filetype': 'cptv10.tsv'
}
print(download_args)

cpt_args = {
    # transformation to apply to the predictand dataset - None, 'Empirical', 'Gamma'
    'transform_predictand': None,
    'tailoring': tailoring,  # tailoring None, 'Anomaly'
    'cca_modes': cca_modes,  # minimum and maximum of allowed CCA modes
    # minimum and maximum of allowed X Principal Componenets
    'x_eof_modes': x_eof_modes,
    # minimum and maximum of allowed Y Principal Components
    'y_eof_modes': y_eof_modes,
    # the type of validation to use; only 'retroactive' is supported for now
    'validation': 'retroactive',
    'drymask': False,  # whether or not to use a drymask of -999
    'scree': scree,  # whether or not to save % explained variance for eof modes
    # number of samples to leave out in each cross-validation step
    'crossvalidation_window': 5,
    # whether or not we are using 'synchronous predictors'
    'synchronous_predictors': True,
    # percent of samples to be used as initial training period for retroactive validation
    'retroactive_initial_training_period': 45,
    # percent of samples to increment retroactive training period by each time
    'retroactive_step': 10,

    # Options for use while debugging:
    'cpt_kwargs': {
        # 'outputdir': 'temp-outputs',  # uncomment to retain CPT output files after it finishes
        # 'interactive': True, # uncomment to see detailed output from CPT
    },
}

print(cpt_args)
force_download = True

[('Week 1', 2, 12), ('Week 2', 13, 19), ('Week 3', 20, 26), ('Week 4', 27, 31)]


NameError: name 'tailoring' is not defined

# Include any models in addition to GEFSv12 (3 URLs for each model)

In [4]:
#subseasonal.hindcasts['ECMWF.PRCP']    = "https://iridl.ldeo.columbia.edu/SOURCES/.ECMWF/.S2S/.ECMF/.reforecast/.perturbed/.sfc_precip/.tp/Y/{predictor_extent['south']}/{predictor_extent['north']}/RANGE/X/{predictor_extent['west']}/{predictor_extent['east']}/RANGE/L/{day1}/{day2}/VALUES/S/7/STEP/S/({training_season}%20{fdate.year})/VALUES/%5BL%5Ddifferences/c%3A//name//water_density/def/998/(kg/m3)/%3Ac/div//mm/unitconvert/-999/setmissing_value/hdate/({fdate.year-20})/({fdate.year-1})/RANGE/dup/%5Bhdate%5Daverage/sub/%5BM%5Daverage/hdate//pointwidth/0/def/-6/shiftGRID/hdate/(days%20since%201960-01-01)/streamgridunitconvert/S/(days%20since%20{fdate.year}-01-01)/streamgridunitconvert/S//units//days/def/L/hdate/add/add/0/RECHUNK/L/removeGRID//name//T/def/2/%7Bexch%5BS/hdate%5D//I/nchunk/NewIntegerGRID/replaceGRIDstream%7Drepeat/use_as_grid/T/grid%3A//name/(T)/def//units/(months%20since%201960-01-01)/def//standard_name/(time)/def//pointwidth/1/def/16/Jan/1700/ensotime/12./16/Jan/2200/ensotime/%3Agrid/replaceGRID//name/(tp)/def//units/(mm)/def//long_name/(precipitation_amount)/def/-999/setmissing_value/{'%5BX/Y%5D%5BT%5Dcptv10.tsv' if filetype == 'cptv10.tsv' else 'data.nc'}"
#subseasonal.obs_template['ECMWF.PRCP'] = "https://iridl.ldeo.columbia.edu/SOURCES/.ECMWF/.S2S/.ECMF/.reforecast/.perturbed/.sfc_precip/.tp/Y/{predictor_extent['south']}/{predictor_extent['north']}/RANGE/X/{predictor_extent['west']}/{predictor_extent['east']}/RANGE/L/{day1}/{day2}/VALUES/S/7/STEP/S/({training_season}%20{fdate.year})/VALUES/%5BL%5Ddifferences/c%3A//name//water_density/def/998/(kg/m3)/%3Ac/div//mm/unitconvert/-999/setmissing_value/hdate/({fdate.year-20})/({fdate.year-1})/RANGE/dup/%5Bhdate%5Daverage/sub/%5BM%5Daverage/hdate//pointwidth/0/def/-6/shiftGRID/hdate/(days%20since%201960-01-01)/streamgridunitconvert/S/(days%20since%20{fdate.year}-01-01)/streamgridunitconvert/S//units//days/def/L/hdate/add/add/0/RECHUNK/L/removeGRID//name//T/def/2/%7Bexch%5BS/hdate%5D//I/nchunk/NewIntegerGRID/replaceGRIDstream%7Drepeat/use_as_grid/{obs_source}/Y/{predictand_extent['south']}/{predictand_extent['north']}/RANGE/X/{predictand_extent['west']}/{predictand_extent['east']}/RANGE/L/{day1}/{day2}/RANGE/T/(days%20since%201960-01-01)/streamgridunitconvert/T/{nday}/runningAverage/{nday}.0/mul/T/2/index/.T/SAMPLE/dup%5BT%5Daverage/sub/-999/setmissing_value/nip/T/grid%3A//name/(T)/def//units/(months%20since%201960-01-01)/def//standard_name/(time)/def//pointwidth/1/def/16/Jan/1700/ensotime/12./16/Jan/2200/ensotime/%3Agrid/replaceGRID//name/(tp)/def//units/(mm)/def//long_name/(precipitation_amount)/def/-999/setmissing_value/{'%5BX/Y%5D%5BT%5Dcptv10.tsv' if filetype == 'cptv10.tsv' else 'data.nc'}"
#subseasonal.forecasts['ECMWF.PRCP']    =   "https://iridl.ldeo.columbia.edu/SOURCES/.ECMWF/.S2S/.ECMF/.forecast/.perturbed/.sfc_precip/.tp/Y/{predictor_extent['south']}/{predictor_extent['north']}/RANGE/X/{predictor_extent['west']}/{predictor_extent['east']}/RANGE/L/{day1}/{day2}/VALUES/S/(0000%20{fdate.day}%20{monthabbrevs[fdate.month]}%20{fdate.year})/VALUE/%5BL%5Ddifferences/%5BM%5Daverage/SOURCES/.ECMWF/.S2S/.ECMF/.reforecast/.perturbed/.sfc_precip/.tp/Y/{predictor_extent['south']}/{predictor_extent['north']}/RANGE/X/{predictor_extent['west']}/{predictor_extent['east']}/RANGE/L/{day1}/{day2}/VALUES/S/(0000%20{fdate.day}%20{monthabbrevs[fdate.month]}%20{fdate.year})/VALUE/%5BL%5Ddifferences/%5BM%5Daverage/%5Bhdate%5Daverage/sub/c%3A//name//water_density/def/998/(kg/m3)/%3Ac/div//mm/unitconvert/grid%3A//name/(T)/def//units/(months%20since%201960-01-01)/def//standard_name/(time)/def//pointwidth/1/def/1/Jan/2261/ensotime/12.0/1/Jan/2261/ensotime/%3Agrid/addGRID/T//pointwidth/0/def/pop//name/(tp)/def//units/(mm)/def//long_name/(precipitation_amount)/def/-999/setmissing_value/{'%5BX/Y%5D%5BT%5Dcptv10.tsv' if filetype == 'cptv10.tsv' else 'data.nc'}"

In [5]:
domain_dir = pycpt.setup(case_dir, download_args["predictor_extent"])

Input data will be saved in /Users/jemal/EDACaP_S2S/2024/Feb/31W-50E_to_-1S-18N/data
Figures will be saved in /Users/jemal/EDACaP_S2S/2024/Feb/31W-50E_to_-1S-18N/figures
Output will be saved in /Users/jemal/EDACaP_S2S/2024/Feb/31W-50E_to_-1S-18N/output


#### Visualize predictor and predictand domains

In [6]:
#pycpt.plot_domains(download_args['predictor_extent'], download_args['predictand_extent'])

#### Download Observations, Hindcasts, and Forecasts from IRI Data Library

In [ ]:
hindcast_data, Y, forecast_data = subseasonal.download_data(predictor_names, predictand_name, download_args, domain_dir, force_download)

URL: https://iridl.ldeo.columbia.edu/SOURCES/.Models/.SubX/.EMC/.GEFSv12_CPC/.hindcast/.weekly/.pr/S/(0000%206%20Jan%201999)/(0000%2028%20Jun%202017)/RANGEEDGES/S/(days%20since%201999-01-01)/streamgridunitconvert/Y/-1/18/RANGE/X/31/50/RANGE/L/1.5/7.5/RANGEEDGES/%5BM%5Daverage/L/7/runningAverage/SOURCES/.Models/.SubX/.EMC/.GEFSv12_CPC/.hindcast/.dc0018/.pr/Y/-1/18/RANGE/L/1.5/7.5/RANGEEDGES/L/7/runningAverage/S/to366daysample/%5BYR%5Daverage/S/sampleDOY/sub/S/(Jan-Mar)/VALUES/L/removeGRID/S/(T)/renameGRID/c%3A/0.001/(m3%20kg-1)/%3Ac/mul/c%3A/1000/(mm%20m-1)/%3Ac/mul/c%3A/7.0//units//days/def/%3Ac/mul/grid%3A//name/(T)/def//units/(months%20since%201960-01-01)/def//standard_name/(time)/def//pointwidth/1/def/16/Jan/1700/ensotime/12./16/Jan/2100/ensotime/%3Agrid/use_as_grid/T//pointwidth/1/def/pop//name/(tp)/def//units/(mm)/def//long_name/(precipitation_amount)/def/-999/setmissing_value/%5BX/Y%5D%5BT%5Dcptv10.tsv

DOWNLOADING: [*************************] (1543 KB) 0:02:25.624015
URL: https:

DOWNLOADING: [*************************] (3066 KB) 0:00:32.360431
URL: https://iridl.ldeo.columbia.edu/SOURCES/.Models/.SubX/.EMC/.GEFSv12_CPC/.forecast/.pr/S/(0000%201%20Feb%202024)/VALUES/Y/-1/18/RANGE/X/31/50/RANGE/L/1.5/7.5/RANGEEDGES/%5BM%5Daverage/L/7/runningAverage/c%3A/86400/(s%20day-1)/%3Ac/mul/SOURCES/.Models/.SubX/.EMC/.GEFSv12_CPC/.hindcast/.dc0018/.pr/Y/-1/18/RANGE/X/31/50/RANGE/L/1.5/7.5/RANGEEDGES/L/7/runningAverage/S/(T)/renameGRID/pentadAverage/pentadmean/T/(S)/renameGRID/%5BS%5DregridLinear/S/1/setgridtype/pop/S/2/index/.S/SAMPLE/sub/c%3A/0.001/(m3%20kg-1)/%3Ac/mul/c%3A/1000/(mm%20m-1)/%3Ac/mul/c%3A/7.0//units//days/def/%3Ac/mul/S/(T)/renameGRID/grid%3A//name/(T)/def//units/(months%20since%201960-01-01)/def//standard_name/(time)/def//pointwidth/1/def/16/Jan/2261/ensotime/12.0/16/Jan/2261/ensotime/%3Agrid/use_as_grid/T//pointwidth/0/def/pop//name/(tp)/def//units/(mm)/def//long_name/(precipitation_amount)/def/-999/setmissing_value/%5BX/Y%5D%5BT%5Dcptv10.tsv

DOWNLOADING

#### Perform CPT Analysis

In [ ]:
hcsts, fcsts, skill, pxs, pys = subseasonal.evaluate_models(hindcast_data, forecast_data, Y, MOS, cpt_args, domain_dir)

#### Plot skill of individual models

Deterministic skill metrics:
- pearson
- spearman
- two_alternative_forced_choice
- roc_area_below_normal (Area under ROC curve for Below Normal category)
- roc_area_above_normal (Area under ROC curve for Above Normal category)

Probabilistic skill metrics:
- generalized_roc
- rank_probability_skill_score

In [ ]:
skill_metrics = [
    "pearson",
    "spearman",
    "two_alternative_forced_choice",
    "roc_area_below_normal",
    "roc_area_above_normal",
    "generalized_roc",
    "rank_probability_skill_score"
]

In [ ]:
#subseasonal.plot_skill(skill, MOS, domain_dir, skill_metrics)

#### Plot EOF Modes

In [ ]:
#subseasonal.plot_eof_modes(pxs, pys, MOS, domain_dir)

#### Plot CCA Modes

In [ ]:
#subseasonal.plot_cca_modes(pxs, pys, MOS, domain_dir)

#### Plot Forecasts

Colormap to use for deterministic forecast. To see available colormaps, run `get_colors_bars()`.

In [ ]:
det_fcst_cmap = 'DL_PRCP_ANOMALY'
det_fcst_vmin = -15
det_fcst_vmax = 15

In [ ]:
#subseasonal.plot_forecasts(fcsts, predictand_name, MOS, cpt_args, domain_dir, color_bar=det_fcst_cmap, vmin=det_fcst_vmin, vmax=det_fcst_vmax)

In [21]:
### EDACaP Geoserver input data preparation
predictor_names = 'GEFSv12.PRCP'
for wks in fcsts['lead_name'].values:
    data = xr.open_dataset(domain_dir / 'output' / (predictor_names + '-' + wks + '_realtime_cca_forecasts.nc'))
    det_data = data['deterministic']
    det_data.to_netcdf(domain_dir / 'output' /('NextGEN_S2S_deterministic_' + str(download_args["fdate"].strftime("%b")) + "_" + wks.replace(" ","") + '.nc'))

    prob_data = data['probabilistic']

    merged_prob_data = xr.Dataset()
    category_mapping = {'1': 'Below_Normal', '2': 'Normal', '3': 'Above_Normal'}
    for value in prob_data['C'].values:
        selected_data = prob_data.sel(C=value).drop_vars('C')
        new_name = category_mapping[str(value)]
        merged_prob_data[new_name] = selected_data
        merged_prob_data.to_netcdf(domain_dir / 'output' / ('NextGEN_S2S_probablistic_' + str(download_args["fdate"].strftime("%b")) + str(download_args["fdate"].year) +"_" + wks.replace(" ","") + '.nc'))
        
    # Skill score data preparation     
    skillscore = xr.open_dataset(domain_dir / 'output' / (predictor_names + '-' + wks + '_skillscores_cca.nc'))
    skillscore.to_netcdf((domain_dir / 'output' / ('NextGEN_S2S_skillscore_' + str(download_args["fdate"].strftime("%b")) + str(download_args["fdate"].year) +"_" + wks.replace(" ","") + '.nc')))
